# HydroFragments with Digital Earth Australia data (via WaterMask-TSFill)

This notebook documents the **real DEA workflow**: pointing `open_water_cube` at a WaterMask-TSFill Zarr output and running HydroFragments on it. If you have not seen the basic pipeline shape yet, start with `01_quickstart.ipynb` first -- this notebook assumes you already know what `open_water_cube` / `analyze` / `metrics_table` are and focuses on the DEA-specific handoff.

## The handoff: WaterMask-TSFill produces, HydroFragments consumes

HydroFragments does **not** talk to Digital Earth Australia or STAC directly -- that is out of scope for this package (no `odc.stac`/`pystac` dependency anywhere in `hydrofragments`). A separate tool, **WaterMask-TSFill**, owns:

- querying DEA/STAC for source imagery,
- deriving a water/no-water classification per date,
- **gapfilling** the time series (filling cloud/shadow/no-data gaps),
- and exporting the result as a canonical Zarr store.

HydroFragments' job starts *after* that export: `open_water_cube` reads the TSFill Zarr output and turns it into the canonical `(water, valid_obs)` pair every metric in this package is built on.

**HydroFragments itself never gapfills, interpolates, or otherwise fills missing data.** If your input has low coverage, HydroFragments will only ever *recommend* running WaterMask-TSFill first -- it will not attempt to compensate internally. See [`docs/input_format.md`](../docs/input_format.md) for the full adapter contract, sentinel-value table, and the baseline-quality/`gapfill` flag mechanics; this notebook does not repeat that reference material.

## A synthetic stand-in for a real TSFill export

A live DEA/TSFill output is not available at notebook-authoring time, so the cell below builds a small **local, synthetic** `.zarr` store that reproduces TSFill's exact uint8 sentinel signature (see `hydrofragments/io/adapters.py`): a `water_mask` variable with values `0` (dry), `1` (water), `254` (outside AOI), `255` (unobserved).

**In real use, replace this cell with your actual TSFill output path** -- e.g. `open_water_cube("/path/to/my_reach_tsfill.zarr")` -- and skip straight to Section 3 below.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from _fixtures import write_synthetic_tsfill_zarr

tsfill_path = Path("synthetic_tsfill_demo.zarr")
write_synthetic_tsfill_zarr(tsfill_path)
tsfill_path

## 1. Open it -- auto-detection picks the `watermask_tsfill` adapter

Leaving `input_kind=None` (the default) lets `open_water_cube` inspect the data and route it to the right adapter automatically. For a TSFill export this means: variable named `water_mask`, sentinel values decoded, `254`/`255` pixels excluded from both `water` and `valid_obs` (never silently counted as dry or wet). Check `cube.provenance` to confirm which adapter was actually used -- this is the same auditability every `open_water_cube` call gives you, real data or synthetic.

In [ ]:
from hydrofragments import open_water_cube

cube = open_water_cube(tsfill_path)
dict(cube.provenance)

Note that `valid_obs` correctly excludes the sentinel pixels -- the "outside AOI" and "unobserved" corner pixels baked into this fixture are never treated as valid, real observations.

In [ ]:
print("outside-AOI pixel ever valid:", bool(cube.valid_obs.isel(y=0, x=0).any()))
print("unobserved pixel ever valid:", bool(cube.valid_obs.isel(y=0, x=1).any()))

## 2. Configure and run `analyze` -- same call as always

Nothing about `analyze` changes based on where the cube came from -- this is the entire point of the `WaterCube` abstraction. Because TSFill already gapfilled this input upstream, we set `gapfill: true` in the config: this tells HydroFragments to trust that declaration and suppress the baseline-quality gapfill *recommendation* it would otherwise surface for low-coverage input (HydroFragments does not re-verify the declaration -- see `docs/input_format.md`).

In [ ]:
from hydrofragments import HydroConfig, analyze

config = HydroConfig.from_mapping(
    {
        "config_schema_version": "1.0.0",
        "input": {"kind": "watermask_tsfill"},
        "temporal": {
            "input_cadence": "monthly",
            "monthly_composite": "supplied",
            "composite_owner": "upstream",
        },
        "gapfill": True,
        "output": {"output_dir": "dea_via_tsfill_out"},
    }
)
result = analyze(cube, aoi_id="dea_demo_reach", config=config, pixel_size_m=30.0)
result.metrics_table[["date", "metric", "value", "unit"]].head(10)

## 3. Check the run manifest for warnings

Every `analyze()` call writes a run manifest recording exactly what ran, including any warnings. With `gapfill: true` declared above, expect **no** gapfill recommendation here, in contrast to what you would see running the same low-coverage input with `gapfill: false` (the default) -- try flipping the flag yourself against a real TSFill output to see the difference.

In [ ]:
import json

manifest_path = Path(result.manifest["run_manifest"])
manifest = json.loads(manifest_path.read_text())
manifest.get("warnings", [])

## Next steps

- **Understanding every metric family:** `03_metrics_walkthrough.ipynb`.
- **Adapter contract details** (sentinel tables, auto-detection rules, the full `gapfill`/baseline-quality mechanics): [`docs/input_format.md`](../docs/input_format.md).
- **Command line, once you have a real config:**
  ```bash
  hydrofragments analyze --config cfg.yaml --input my_reach_tsfill.zarr --aoi my_reach --out results/
  ```